Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 12 — wykrywanie anomalii za pomocą autoenkodera

Materiał towarzyszący [`README.pl.md`](README.pl.md). Zbiór: Fashion-MNIST, klasa
normalna = *Sneaker* (7), anomalie = pozostałe 9 klas. **Nie** to samo
zadanie, co w wykładzie (tam: KMNIST, rekonstrukcja/generacja) — tu: detekcja
anomalii, z naciskiem na to, kiedy ta metoda zawodzi.

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt  # noqa: F401 - uzywane w TODO ponizej
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as tvd
import torchvision.transforms as T

_REPO_ROOT = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "pyproject.toml").is_file()
    ),
    Path.cwd(),
)
DATA_CACHE_DIR = Path(
    os.environ.get("DEEP_LEARNING_DATA_CACHE", _REPO_ROOT / "data_cache")
).expanduser().resolve()

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
NORMAL_CLASS = 7  # Sneaker

train_full = tvd.FashionMNIST(
    root=DATA_CACHE_DIR, train=True, download=True, transform=T.ToTensor()
)
test_full = tvd.FashionMNIST(
    root=DATA_CACHE_DIR, train=False, download=True, transform=T.ToTensor()
)

train_imgs = torch.stack([train_full[i][0] for i in range(len(train_full))])
train_labels = torch.tensor([train_full[i][1] for i in range(len(train_full))])
test_imgs = torch.stack([test_full[i][0] for i in range(len(test_full))])
test_labels = torch.tensor([test_full[i][1] for i in range(len(test_full))])

print(f"train: {train_imgs.shape}, test: {test_imgs.shape}")
print(f"klasa normalna: {NORMAL_CLASS} ({CLASS_NAMES[NORMAL_CLASS]})")

D_IN = 28 * 28

## 1. Dane i podział — Twoje zadanie

Zbuduj **pięć** rozłącznych podzbiorów (zobacz `README.pl.md`, sekcja 1, dla
pełnych instrukcji):

1. `ae_train_X` — tylko klasa normalna, z `train_imgs`/`train_labels`.
2. `val_normal_X` — **inne** przykłady klasy normalnej, z `train_imgs`
   (rozłączne z `ae_train_X`).
3. `val_anomaly_X` — próbka innych klas, z `train_imgs`.
4. `test_normal_X` — klasa normalna, z `test_imgs`/`test_labels` (osobny,
   oficjalny zbiór testowy).
5. `test_anomaly_X` (+ `test_anomaly_labels` — przydatne do analizy błędów
   per klasa w sekcji 6) — próbka innych klas, z `test_imgs`.

**Zweryfikuj programowo**, że `ae_train_X`/`val_normal_X`/`val_anomaly_X` nie
nakładają się indeksami (nie tylko porównaj rozmiary).

In [ ]:
# TODO:
# ae_train_X = ...      # tylko klasa normalna, z train_imgs (podzbior 1)
# val_normal_X = ...    # klasa normalna, INNE przykłady niz ae_train_X (podzbior 2)
# val_anomaly_X = ...   # inne klasy, z train_imgs (podzbior 3)
# test_normal_X = ...   # klasa normalna, z test_imgs (podzbior 4)
# test_anomaly_X = ...  # inne klasy, z test_imgs
# test_anomaly_labels = ...  # odpowiadajace etykiety klas (do analizy w sekcji 6)
# wypisz rozmiary, ZWERYFIKUJ PROGRAMOWO brak nakladania ae_train/val_normal/val_anomaly
raise NotImplementedError("TODO: podzial danych bez przecieku")

## 2. Autoenkoder — architektura i trening (gotowe)

Mały AE (MLP), tak jak w wykładzie. Trenujemy **wyłącznie** na danych
normalnych — to jest kluczowa zasada tego zadania, nie szczegół
implementacyjny.

In [ ]:
class AE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(D_IN, 128), nn.ReLU(), nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(), nn.Linear(128, D_IN), nn.Sigmoid(),
        )

    def forward(self, x):
        e = self.encoder(x)
        return self.decoder(e).view(-1, 1, 28, 28), e


def train_ae(train_X, latent_dim=16, epochs=15, batch_size=64, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    model = AE(latent_dim)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    n = train_X.shape[0]
    t0 = time.time()
    for _ in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            batch = train_X[idx]
            opt.zero_grad()
            x_hat, _ = model(batch)
            loss = F.mse_loss(x_hat, batch)
            loss.backward()
            opt.step()
    model.eval()
    print(f"czas treningu: {time.time() - t0:.2f}s")
    return model

In [ ]:
# TODO: wytrenuj AE WYLACZNIE na ae_train_X (NIE na train_imgs, NIE na polaczeniu
# z czymkolwiek innym) -- uzyj train_ae(...) z sekcji 2 (gotowe), latent_dim=16.
# model = train_ae(???)
raise NotImplementedError("TODO: trening AE na danych normalnych")

## 3. Błąd rekonstrukcji i rozkłady — Twoje zadanie

Zaimplementuj `per_image_error(model, X)` -- błąd rekonstrukcji **per obraz**
(nie zagregowany): `((x_hat - x)**2).flatten(1).mean(dim=1)`.
Policz dla `val_normal_X` i `val_anomaly_X`, wypisz mean/std/median, narysuj
histogram obu rozkładów na jednym wykresie.

In [ ]:
# TODO:
# def per_image_error(model, X):
#     ...
# err_val_normal = per_image_error(model, val_normal_X)
# err_val_anomaly = per_image_error(model, val_anomaly_X)
# wypisz mean/std/median dla obu, narysuj histogram
raise NotImplementedError("TODO: blad rekonstrukcji i rozklady")

## 4. Wybór progu — Twoje zadanie

Ustaw próg **wyłącznie** na podstawie `err_val_normal` (np. mean + k*std dla
wybranego przez Ciebie k -- uzasadnij). **Nie** używaj `err_val_anomaly`.

In [ ]:
# TODO: threshold = err_val_normal.mean() + k * err_val_normal.std()  (wybierz k, uzasadnij)
raise NotImplementedError("TODO: wybor progu z val_normal")

## 5. Metryki — Twoje zadanie

**Najpierw** zweryfikuj swoją implementację na znanym przykładzie (zobacz
`README.pl.md`, sekcja 5): `scores`/`labels` poniżej, próg $0{,}45$. Oczekiwane:
TP=5, FP=1, TN=4, FN=0, precision≈0,833, recall=1,0, F1≈0,909.

In [ ]:
known_scores = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.9, 0.5, 0.6, 0.7, 0.8, 1.0])
known_labels = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])
known_threshold = 0.45

# TODO:
# def confusion_counts(scores, labels, threshold):  # -> tp, fp, tn, fn
#     ...
# def precision_recall_f1(tp, fp, fn):
#     ...
# zweryfikuj na known_scores/known_labels/known_threshold, PRZED uzyciem na prawdziwych danych
raise NotImplementedError("TODO: metryki na znanym przykladzie")

In [ ]:
# TODO: zaimplementuj recznie (sklearn NIE jest zainstalowany w tym repo):
# def roc_auc(labels, scores):       # rank-based: P(score_anomalia > score_normalny)
#     ...
# def pr_auc(labels, scores):        # average precision
#     ...
# zweryfikuj obie na known_scores/known_labels (oczekiwane: roc_auc=0.84, pr_auc=0.81)
raise NotImplementedError("TODO: ROC-AUC / PR-AUC na znanym przykladzie")

In [ ]:
# TODO: policz err_test_normal, err_test_anomaly (per_image_error z sekcji 3),
# potem confusion_counts/precision_recall_f1/roc_auc/pr_auc na PRAWDZIWYCH danych
# testowych, uzywajac threshold z sekcji 4 (NIGDY nie wybieraj progu tutaj!)
raise NotImplementedError("TODO: metryki na danych testowych")

## 6. Analiza błędów i prosty baseline — Twoje zadanie

**Baseline bez uczenia**: błąd = odległość do średniego obrazu klasy
normalnej (`ae_train_X.mean(dim=0)`), bez żadnego treningu. Policz te same
metryki dla tego baseline na danych testowych i porównaj z AE -- **bez**
zakładania z góry, który wypadnie lepiej.

In [ ]:
# TODO:
# mean_normal_image = ae_train_X.mean(dim=0, keepdim=True)
# baseline_err_test_normal = ((test_normal_X - mean_normal_image) ** 2).flatten(1).mean(dim=1)
# baseline_err_test_anomaly = ((test_anomaly_X - mean_normal_image) ** 2).flatten(1).mean(dim=1)
# policz roc_auc dla baseline, porownaj z AE roc_auc z sekcji 5
raise NotImplementedError("TODO: trywialny baseline")

In [ ]:
# TODO: znajdz PRZYNAJMNIEJ jeden konkretny przypadek:
# 1. anomalia z NISKIM bledem rekonstrukcji (false negative) -- jakiej klasy? wizualizuj.
# 2. przykald normalny z WYSOKIM bledem (false positive) -- wizualizuj.
# (test_anomaly_labels przydatne do sprawdzenia, z jakiej klasy pochodzi false negative)
raise NotImplementedError("TODO: analiza bledow")

## 7. Zadanie diagnostyczne: przeciek danych w wyborze progu

Poniższy kod (gotowy, uruchom go) **wygląda** rozsądnie -- przeszukuje
siatkę progów i wybiera tę o najlepszym F1 -- ale wybiera **na podstawie
etykiet ze zbioru testowego**.

In [ ]:
# noqa: F821 - err_test_normal/err_test_anomaly/confusion_counts/precision_recall_f1
# zdefiniowane przez Ciebie w sekcjach 3 i 5
candidate_thresholds = torch.linspace(err_test_normal.min(), err_test_anomaly.max(), 500)  # noqa: F821
best_f1_leaky, best_t_leaky = -1.0, None
for t in candidate_thresholds:
    tp_, fp_, tn_, fn_ = confusion_counts(  # noqa: F821
        torch.cat([err_test_normal, err_test_anomaly]),  # noqa: F821
        torch.cat([torch.zeros(len(err_test_normal)), torch.ones(len(err_test_anomaly))]),  # noqa: F821
        t.item(),
    )
    _, _, f1_ = precision_recall_f1(tp_, fp_, fn_)  # noqa: F821
    if f1_ > best_f1_leaky:
        best_f1_leaky, best_t_leaky = f1_, t.item()

print(f"'przeciekowy' prog (wybrany na etykietach testowych): {best_t_leaky:.5f}")
print(f"'przeciekowe' F1 (na tym samym zbiorze, ktory wybral prog): {best_f1_leaky:.4f}")

In [ ]:
# TODO:
# 1. porownaj best_t_leaky z Twoim threshold z sekcji 4 -- identyczne?
# 2. porownaj best_f1_leaky z Twoim honest F1 z sekcji 5 (na tescie, z progiem z sekcji 4)
# 3. w komentarzu: wyjasnij, dlaczego ta procedura jest bledna NIEZALEZNIE od tego,
#    czy roznica F1 wyszla duza czy mala w tym konkretnym przebiegu
raise NotImplementedError("TODO: diagnoza przecieku danych")

## 8. Wnioski — Twoje zadanie

**1. Jak dobrze (konkretne liczby) Twój AE rozdziela klasę normalną od anomalii na zbiorze testowym?**

_(Twoja odpowiedź tutaj.)_

**2. Jak wypada w porównaniu z trywialnym baseline'em? Co to mówi o tym, skąd bierze się zdolność rozróżniania?**

_(Twoja odpowiedź tutaj.)_

**3. Jaki jest Twój najbardziej przekonujący konkretny przykład ograniczenia metody (sekcja 6)?**

_(Twoja odpowiedź tutaj.)_

**4. Jaki błąd wprowadził przeciek danych w sekcji 7, i jaka jest ogólna zasada, którą z tego wyciągasz?**

_(Twoja odpowiedź tutaj.)_

**5. Czy uznałbyś/uznałabyś ten system za gotowy do praktycznego zastosowania? Czego by jeszcze potrzebował?**

_(Twoja odpowiedź tutaj.)_

## Rozszerzenie (opcjonalne)

Zobacz `README.pl.md` -- wpływ wymiaru latentnego albo VAE jako alternatywa.
Nie wliczone w budżet 115 minut.